## Reconciliation

This notebook validates the migration by comparing source and target state.

**What it checks:**
1. Deployment status for each space
2. Benchmark counts match
3. Permissions applied
4. Data sources accessible

**Output:**
- Reconciliation report CSV
- Summary with pass/fail status

## Install Dependencies

In [ ]:
%pip install databricks-sdk>=0.76.0 --quiet
dbutils.library.restartPython()

## Setup Path Resolution

In [ ]:
import sys
import os

notebook_path = os.getcwd()
if notebook_path.startswith("/Workspace"):
    bundle_root = os.path.dirname(notebook_path)
else:
    bundle_root = os.path.dirname(os.path.abspath("__file__"))

helpers_path = os.path.join(bundle_root, "helpers")
if not os.path.exists(helpers_path):
    helpers_path = os.path.join(os.path.dirname(bundle_root), "helpers")

if helpers_path not in sys.path:
    sys.path.insert(0, os.path.dirname(helpers_path))

print(f"Bundle root: {bundle_root}")

## Read Parameters

In [ ]:
dbutils.widgets.text("volume_path", "", "Import Volume Path")

volume_path = dbutils.widgets.get("volume_path")

if not volume_path:
    raise ValueError("volume_path parameter is required")

print(f"Volume path: {volume_path}")

## Load Deployment Manifest

In [ ]:
import csv

manifest_path = os.path.join(volume_path, "deployment_manifest.csv")

if not os.path.exists(manifest_path):
    raise FileNotFoundError(
        f"Deployment manifest not found: {manifest_path}\n"
        "Run Tgt_02_Deploy_Genie_Spaces first."
    )

with open(manifest_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    deployment_results = list(reader)

print(f"Loaded {len(deployment_results)} deployment results")

## Generate Reconciliation Report

In [ ]:
from databricks.sdk import WorkspaceClient
from helpers.reconciliation import (
    generate_reconciliation_report,
    write_reconciliation_report,
    generate_reconciliation_summary
)

w = WorkspaceClient()
print(f"Connected to: {w.config.host}")

print("\nGenerating reconciliation report...")
entries = generate_reconciliation_report(
    client=w,
    deployment_results=deployment_results,
    import_path=volume_path
)

print(f"Processed {len(entries)} space(s)")

## Display Results

In [ ]:
import pandas as pd

df = pd.DataFrame([e.to_dict() for e in entries])
display(df)

## Write Report

In [ ]:
import json

report_path = os.path.join(volume_path, "reconciliation_report.csv")
write_reconciliation_report(entries, report_path)
print(f"Report written to: {report_path}")

summary = generate_reconciliation_summary(entries)

summary_path = os.path.join(volume_path, "reconciliation_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(f"Summary written to: {summary_path}")

## Summary

In [ ]:
print("=" * 60)
print("RECONCILIATION COMPLETE")
print("=" * 60)
print(f"Total spaces:        {summary['total_spaces']}")
print(f"Successful:          {summary['successful']}")
print(f"Partial:             {summary['partial']}")
print(f"Failed:              {summary['failed']}")
print(f"")
print(f"Source benchmarks:   {summary['total_source_benchmarks']}")
print(f"Target benchmarks:   {summary['total_target_benchmarks']}")
print(f"Benchmarks match:    {'✓' if summary['benchmarks_preserved'] else '✗'}")
print(f"Data sources valid:  {'✓' if summary['data_sources_valid'] else '✗'}")
print("")

if summary['failed'] == 0 and summary['partial'] == 0:
    print("✓ MIGRATION SUCCESSFUL")
elif summary['failed'] > 0:
    print("✗ MIGRATION HAS FAILURES - review the report above")
else:
    print("⚠️  MIGRATION PARTIALLY SUCCESSFUL - some issues detected")

print(f"\nFull report: {report_path}")

# Fail the job if there are deployment failures
if summary['failed'] > 0:
    failed_spaces = [e.source_title for e in entries if e.overall_status == "FAILED"]
    raise Exception(
        f"Migration failed for {summary['failed']} space(s): {', '.join(failed_spaces)}. "
        f"Check deployment_manifest.csv and reconciliation_report.csv for details."
    )